In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    print('cuda')

cuda


In [2]:
torch.cuda.empty_cache()

In [3]:
model_path = "./my_4bit_model"  
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
non_quanitzed_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True  
)

Loading weights: 100%|██████████| 355/355 [00:15<00:00, 22.43it/s]


In [7]:
prompt = "Language modeling is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

# Decode and print the response
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)


Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 355/355 [00:17<00:00, 20.75it/s]


Language modeling is one of the key components of most deep learning architectures (BIBREF14). There are various types of sequence models that are able to predict the next word given a context word or phrase. These models are based on neural network architectures such as long short-term memory (LSTM) (BIBREF29) and gated recurrent units (GRU) (BIBREF30). The choice of neural model architecture depends on the use case of the sequence modeling: for text applications, a LSTM is the


In [4]:
# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


In [7]:
bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True,
    dtype=torch.float16
)


Loading weights: 100%|██████████| 355/355 [00:11<00:00, 30.52it/s]


In [6]:
memory_bytes = model.get_memory_footprint()
memory_gb = memory_bytes / (1024 ** 3)
print(f"Model memory footprint: {memory_gb:.2f} GB")

Model memory footprint: 4.55 GB


In [7]:
model.save_pretrained("./my_4bit_model")

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.16s/it]


In [9]:
import torch
import psutil
import os

def get_model_size(model):
    param_count = sum(p.numel() for p in model.parameters())
    param_size_bytes = param_count * 2 
    
    buffer_size = 0
    if hasattr(model, 'model') and hasattr(model.model, 'buffers'):
        buffer_size = sum(b.numel() for b in model.model.buffers()) * 2
    
    total_size_gb = (param_size_bytes + buffer_size) / (1024**3)
    
    return {
        "parameters": f"{param_count:,}",
        "parameters_billions": param_count / 1e9,
        "estimated_size_gb": total_size_gb
    }

size_info = get_model_size(model)
print(f"Model parameters: {size_info['parameters']}")
print(f"Model size (billions): {size_info['parameters_billions']:.2f}B")
print(f"Estimated model size: {size_info['estimated_size_gb']:.2f} GB")

Model parameters: 4,060,614,656
Model size (billions): 4.06B
Estimated model size: 7.56 GB


In [11]:

prompt = "What is a checking account?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)


What is a checking account? What is a savings account? What is the difference between the two? These are questions that most of us have probably asked at some point or another.

The truth is, the difference between a checking account and a savings account is fairly simple. The major difference between the two is the way that each account is used and the way that money is withdrawn from the account. However, there are some other differences that you should be aware of before you decide on a checking or savings account.

What is a Checking Account?

A checking account is a deposit account held at a bank or other financial institution. This account allows you to write checks or use a debit card to make purchases. It is important to note that checking accounts do not usually earn any interest.

This account is designed to be a way to pay for items that you need to purchase. Therefore, it is often the account that is used to pay for things like bills or groceries. In order to make payments,

In [29]:
user_input = "Checking Account भएको नेपाली भाषामा, चेकिङ खातालाई सामान्यतया करेन्ट एकाउन्ट वा साधारण रूपमा खाता भनिन्छ। यद्यपि अङ्ग्रेजी शब्द ‘चेकिङ एकाउन्ट’ प्रायः बैंकिङ सन्दर्भमा प्रयोग गरिन्छ, ‘एकाउन्ट’ को प्रत्यक्ष अनुवाद खाता हो, र बारम्बार जम्मा तथा निकासीको लागि लेनदेन खाताको अवधारणा यसै शब्द मार्फत बुझिन्छ।"

context = "चेकिङ खातालाई सामान्यतया करेन्ट एकाउन्ट वा साधारण रूपमा खाता भनिन्छ। यद्यपि अङ्ग्रेजी शब्द 'चेकिङ एकाउन्ट' प्रायः बैंकिङ सन्दर्भमा प्रयोग गरिन्छ, 'एकाउन्ट' को प्रत्यक्ष अनुवाद खाता हो, र बारम्बार जम्मा तथा निकासीको लागि लेनदेन खाताको अवधारणा यसै शब्द मार्फत बुझिन्छ।"

# First, detect if the user is asking a relevant question
prompt = f"""### System Instructions:
You are a helpful Nepali banking assistant. Your task:

1. If the user's question is **unclear, nonsensical, or contains unrelated content** (like family members, princes, votes), politely ask them to rephrase their question about banking.

2. If the question is **relevant to banking**, answer based ONLY on the context provided.

3. Respond in **Romanized Nepali** (Nepali written with English/Latin letters) to match the user's writing style.

4. Be concise and helpful.

### Context (Banking Information in Nepali):
{context}

### User Question (Romanized Nepali):
{user_input}

### Analysis:
First, determine if this question is about banking or unrelated.

### Response:
"""

# Generation with appropriate settings
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,  # Enable sampling for more natural responses
    temperature=0.3,  # Low temperature for focused responses
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the response part
if "### Response:" in response:
    final_output = response.split("### Response:")[-1].strip()
else:
    final_output = response.strip()

print(final_output)

चिनाई खत भेटिने राम्रो विचार हो भरिएका छैनन्, तर आफ्नो खर्च लगाउँछौं भिनेको कुरा आर्थिक रुपमै समाधिन गाह्रो हो।


In [28]:
user_input = "Checking Account भएको छैन भाइ र बहिनी यस्ता गर्ने अवसर छ भतीदार राजकुमारको साथ लाएका छन् तर त्यसको प्रतिकूल अनुभव भिन्न भोट भूमि"

context = "चेकिङ खातालाई सामान्यतया करेन्ट एकाउन्ट वा साधारण रूपमा खाता भनिन्छ। यद्यपि अङ्ग्रेजी शब्द 'चेकिङ एकाउन्ट' प्रायः बैंकिङ सन्दर्भमा प्रयोग गरिन्छ, 'एकाउन्ट' को प्रत्यक्ष अनुवाद खाता हो, र बारम्बार जम्मा तथा निकासीको लागि लेनदेन खाताको अवधारणा यसै शब्द मार्फत बुझिन्छ।"

# First, detect if the user is asking a relevant question
prompt = f"""### System Instructions:
You are a helpful Nepali banking assistant. Your task:

1. If the user's question is **unclear, nonsensical, or contains unrelated content**

2. If the question is **relevant to banking**, answer based ONLY on the context provided.

3. Respond in **Romanized Nepali** (Nepali written with English/Latin letters) to match the user's writing style.

4. Be concise and helpful.

### Context (Banking Information in Nepali):
{context}

### User Question (Romanized Nepali):
{user_input}

### Analysis:
First, determine if this question is about banking or unrelated.

### Response:
"""

# Generation with appropriate settings
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to('cpu')

outputs = non_quanitzed_model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,  # Enable sampling for more natural responses
    temperature=0.3,  # Low temperature for focused responses
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "### Response:" in response:
    final_output = response.split("### Response:")[-1].strip()
else:
    final_output = response.strip()

print(final_output)

यस ब्याख्या लिने काम भर्खर रहेको हो भेटिन र साहिब यो बेला आफ्नो चेनिङ कारोबाडी रख्नु अर्को काहाँ गारी छ ।


In [31]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",         
    trust_remote_code=True,
   
)

Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 355/355 [03:48<00:00,  1.55it/s]


In [ ]:
def bank_query(user_input: str, context: str = "") -> str:
    """
    Correct OLMo-2 inference using manual instruct format.
    Works for English, Nepali, and Romanized Nepali.
    """
    system_prompt = """You are a helpful Nepali banking assistant.
Answer only from the provided context.
If context is missing, redirect to branch.
Reply in the same script as the question."""

    user_content = f"Context:\n{context}\n\nQuestion: {user_input}" \
                   if context else user_input

    # Manual prompt construction (OLMo-2 instruct format)
    prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_content}\n<|assistant|>\n"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
context = """
Bachat khata kholna minimum Rs. 100 chaincha.
Byaj dar barshik 5% cha.
Nagarikta praman patra ra 2 wata photo aawashyak cha.
"""

print(bank_query("bachat khata kholna k chaincha?", context))

# Test 2: Devanagari
print(bank_query("बचत खाता खोल्न के चाहिन्छ?", context))

# Test 3: Out of scope
print(bank_query("aaja mausam kasto cha?"))

# Test 4: Sensitive
print(bank_query("mero balance kati cha?"))

In [1]:
import tiny_llm_scratch_with_tokenizer as m
print(m.__file__)          # confirm which .so is loaded

/home/lang-chain/Documents/tiny_LLM_scratch_with_tokenizer/.venv/lib/python3.11/site-packages/tiny_llm_scratch_with_tokenizer/__init__.py


In [2]:
import sys
import statistics
import time
from tiny_llm_scratch_with_tokenizer import PyNepBPETokenizer

VOCAB_TSV = 'vocab_nepbpe/nepbpe_vocab_bilingual_new.tsv'
#"dataset_ne/nepbpe_vocab_new.tsv"

In [7]:
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

data="""
विद्यालयमा
विद्यालयको
विद्यालयदेखि
विद्यालयसम्म
विद्यालयबाट
"""
tok = PyNepBPETokenizer(folding_rules=FOLDING_RULES)

# ----- ADD THIS LINE -----
tok.load_vocab_tsv(VOCAB_TSV)
# --------------------------

print("id:", tok.vocab_get_id(data))  
print("size:", tok.vocab_size())           

ids = tok.encode(data)
print("surfaces:", [tok.get_token_surface(i) for i in ids])


id: None
size: 48001
surfaces: ['▁', 'Ċ', 'विद्यालय', 'मा', 'Ċ', 'विद्यालय', 'को', 'Ċ', 'विद्यालय', 'देखि', 'Ċ', 'विद्यालय', 'सम्म', 'Ċ', 'विद्यालय', 'बाट', 'Ċ']


In [1]:
import tiny_llm_scratch_with_tokenizer as m
print(m.__file__)          

/home/lang-chain/Documents/tiny_LLM_scratch_with_tokenizer/.venv/lib/python3.11/site-packages/tiny_llm_scratch_with_tokenizer/__init__.py


In [ ]:
for word in data.split():
    ids = tok.encode(word)
    print(word, "->", [tok.get_token_surface(i) for i in ids])
    

विद्यालयमा -> ['▁विद्यालयमा']
विद्यालयको -> ['▁विद्यालयको']
विद्यालयदेखि -> ['▁विद्यालय', 'देखि']
विद्यालयसम्म -> ['▁विद्यालय', 'सम्म']
विद्यालयबाट -> ['▁विद्यालयबाट']


In [ ]:
ids = tok.encode("सन् 2020 मा गा.वि.स.को निर्णय")
surfaces = [tok.get_token_surface(i) for i in ids]

surfaces
['▁', 'सन्', '▁', '2020', '▁', 'मा', '▁', 'गा.वि.स.', 'को', '▁निर्णय']

['▁', 'सन्', '▁', '2020', '▁', 'मा', '▁', 'गा.वि.स.', 'को', '▁निर्णय']

In [11]:
corpus = """Let me be direct about what arXiv is and isn't, because it matters for your goal. arXiv is not peer-reviewed — anyone endorsed in the category can post. So "publishable on arXiv" is almost always yes. But that also means arXiv is where over-claimed work goes to be quietly ignored or publicly picked apart. Your contribution to the community is maximized not by posting, but by posting something people trust and reuse. The 100%-coverage/zero-UNK/Nepali tokenizer is genuinely useful — Nepali is under-served, and a fast, complete, open tokenizer is a real gift to that community. So the work clears the "worth sharing" bar easily. The only thing standing between you and a good contribution is the honesty punch-list from my last two messages."""

In [5]:
from HimalTokWrapper import HimalayanTokenizer

TOKENIZER_DIR = "./my-nepali-tokenizer"

tokenizer = HimalayanTokenizer.from_pretrained(TOKENIZER_DIR)
print("vocab_size:", tokenizer.vocab_size)
print("cls_token_id:", tokenizer.cls_token_id)
print("sep_token_id:", tokenizer.sep_token_id)
print("pad_token_id:", tokenizer.pad_token_id)
print("unk_token_id:", tokenizer.unk_token_id)
print("mask_token_id:", tokenizer.mask_token_id)

vocab_size: 64014
cls_token_id: 64010
sep_token_id: 64011
pad_token_id: 64012
unk_token_id: 64009
mask_token_id: 64013


In [6]:
ids = tokenizer.encode(corpus, add_special_tokens=True)
tokens = tokenizer.convert_ids_to_tokens(ids)
decoded = tokenizer.decode(ids, skip_special_tokens=True)

print("corpus:   ", corpus)
print("ids:    ", ids)
print("tokens: ", tokens)
print("decoded:", decoded)
print("roundtrip ok:", decoded.strip() == corpus.strip())

corpus:    Let me be direct about what arXiv is and isn't, because it matters for your goal. arXiv is not peer-reviewed — anyone endorsed in the category can post. So "publishable on arXiv" is almost always yes. But that also means arXiv is where over-claimed work goes to be quietly ignored or publicly picked apart. Your contribution to the community is maximized not by posting, but by posting something people trust and reuse. The 100%-coverage/zero-UNK/Nepali tokenizer is genuinely useful — Nepali is under-served, and a fast, complete, open tokenizer is a real gift to that community. So the work clears the "worth sharing" bar easily. The only thing standing between you and a good contribution is the honesty punch-list from my last two messages.
ids:     [64010, 55415, 49082, 48372, 54099, 48177, 49791, 55908, 571, 44043, 48146, 48038, 57876, 491, 541, 479, 51862, 48388, 54213, 540, 48000, 44032, 48000, 46105, 539, 48000, 46036, 478, 55908, 571, 44043, 48146, 48538, 49712, 44002, 493, 

In [7]:
text = "नेपालको संविधान २०७२ मा जारी भएको थियो"

ids = tokenizer.encode(text, add_special_tokens=True)
tokens = tokenizer.convert_ids_to_tokens(ids)
decoded = tokenizer.decode(ids, skip_special_tokens=True)

print("text:   ", text)
print("ids:    ", ids)
print("tokens: ", tokens)
print("decoded:", decoded)
print("roundtrip ok:", decoded.strip() == text.strip())

text:    नेपालको संविधान २०७२ मा जारी भएको थियो
ids:     [64010, 1622, 1607, 4664, 810, 1725, 857, 938, 64011]
tokens:  ['[CLS]', '▁नेपालको', '▁संविधान', '▁२०७२', '▁मा', '▁जारी', '▁भएको', '▁थियो', '[SEP]']
decoded: ▁नेपालको ▁संविधान ▁२०७२ ▁मा ▁जारी ▁भएको ▁थियो
roundtrip ok: False


In [13]:
import sys
import statistics
import time

from HimalayanTOK_Nepali_64K import PyHimalayanTOK_Nepali_64K
VOCAB_TSV = "vocab_nepbpe/vocab_v4.tsv"

# MUST be identical to what you trained with (train.py). I4f these differ,
# normalization drifts and surface lookups miss.
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

# 'Ġ' (U+0120) is the byte-alphabet surface for space (0x20). Without
# Ġ-prefixing, each inter-word space is its own token.
SPACE_PIECE = "\u0120"


def show_piece(p: str) -> str:
    """Render a piece for display: space as ·, ZWNJ as <ZWNJ>."""
    if p == SPACE_PIECE:
        return "·"
    if p == "\u200c":
        return "<ZWNJ>"
    return p


SAMPLES = corpus


def main(test_file=None) -> None:
    tok = PyHimalayanTOK_Nepali_64K(folding_rules=FOLDING_RULES)
    n = tok.load_vocab_tsv(VOCAB_TSV)
    print(f"loaded {n} tokens from {VOCAB_TSV}\n")

    raw_rates, content_rates = [], []
    tok_total = word_total = space_total = fails = 0

    print("=== sample tokenization ===")
    for idx, s in enumerate(SAMPLES, 1):
        print(f"  [{idx}/{len(SAMPLES)}] processing...", end="", flush=True)

        ids = tok.encode(s)
        pieces = [tok.get_token_surface(i) for i in ids]
        norm = tok.normalize(s)
        words = max(1, len(norm.split()))
        spaces = sum(1 for p in pieces if p == SPACE_PIECE)
        content = len(ids) - spaces
        ok = tok.decode(ids) == norm

        raw_rates.append(len(ids) / words)
        content_rates.append(content / words)
        tok_total += len(ids)
        word_total += words
        space_total += spaces
        if not ok:
            fails += 1

        shown = " ".join(show_piece(p) for p in pieces)
        print(f"\r  {s}")
        print(
            f"    {len(ids)} tok = {content} content + {spaces} space | "
            f"{len(ids)/words:.2f}/word ({content/words:.2f} ex-space) | "
            f"roundtrip={'OK' if ok else 'FAIL'}"
        )
        print(f"    {shown}")
        if not ok:
            print(f"    DECODED : {tok.decode(ids)!r}")
            print(f"    EXPECTED: {norm!r}")

    print("\n=== sample summary ===")
    print(
        f"  tokens/word   : mean={statistics.mean(raw_rates):.2f}  "
        f"median={statistics.median(raw_rates):.2f}"
    )
    print(
        f"  ex-space/word : mean={statistics.mean(content_rates):.2f}  "
        f"median={statistics.median(content_rates):.2f}   <- real subword fertility"
    )
    print(
        f"  micro/word    : {tok_total/max(1,word_total):.2f}  "
        f"(space tokens = {space_total}/{tok_total} = "
        f"{100*space_total/max(1,tok_total):.0f}%)"
    )
    print(f"  roundtrip     : {len(SAMPLES)-fails}/{len(SAMPLES)} OK")

    # Optional: fertility over a held-out file (fast, uses the Rust encode path).
    if test_file:
        print(f"\n=== fertility over {test_file} ===")
        space_id = tok.vocab_get_id(SPACE_PIECE)  # int (or None), computed once
        tt = ww = ss = lines = 0
        t0 = time.perf_counter()
        try:
            with open(test_file, encoding="utf-8") as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue

                    if line_num % 1000 == 0:
                        elapsed = time.perf_counter() - t0
                        print(
                            f"  ... line {line_num}: {tt} tokens in {elapsed:.1f}s "
                            f"({tt/max(1,elapsed):.0f} tok/s)",
                            flush=True,
                        )

                    w = len(tok.normalize(line).split())
                    if w == 0:
                        continue

                    ids = tok.encode(line)
                    sp = ids.count(space_id) if space_id is not None else 0
                    tt += len(ids)
                    ss += sp
                    ww += w
                    lines += 1

                    if lines >= 20000:
                        print(f"  Reached {lines} lines limit", flush=True)
                        break

        except FileNotFoundError:
            print(f"  Error: File '{test_file}' not found. Skipping fertility analysis.")
            return
        except KeyboardInterrupt:
            print(f"\n  Interrupted after {lines} lines", flush=True)
            return

        dt = time.perf_counter() - t0
        print(
            f"  lines={lines} | tokens={tt} | tokens/word={tt/max(1,ww):.3f} | "
            f"ex-space/word={(tt-ss)/max(1,ww):.3f} | space-frac={ss/max(1,tt):.3f} | "
            f"{dt:.1f}s ({tt/max(1,dt):.0f} tok/s)"
        )


if __name__ == "__main__":
    # Handle both command-line and Jupyter environments.
    try:
        if len(sys.argv) > 1 and not sys.argv[1].startswith("--f="):
            main(sys.argv[1])
        else:
            main()
    except KeyboardInterrupt:
        print("\nInterrupted by user", file=sys.stderr)

loaded 64014 tokens from vocab_nepbpe/vocab_v4.tsv

=== sample tokenization ===
  L1/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1.00 ex-space) | roundtrip=OK
    ▂L
  e2/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1.00 ex-space) | roundtrip=OK
    ▂e
  t3/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1.00 ex-space) | roundtrip=OK
    ▂t
   4/744] processing...
    2 tok = 2 content + 0 space | 2.00/word (2.00 ex-space) | roundtrip=OK
    ▁ ▁
  m5/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1.00 ex-space) | roundtrip=OK
    ▂m
  e6/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1.00 ex-space) | roundtrip=OK
    ▂e
   7/744] processing...
    2 tok = 2 content + 0 space | 2.00/word (2.00 ex-space) | roundtrip=OK
    ▁ ▁
  b8/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1.00 ex-space) | roundtrip=OK
    ▂b
  e9/744] processing...
    1 tok = 1 content + 0 space | 1.00/word (1

In [19]:
# Jupyter notebook cell
import sys
import statistics
import time

from HimalayanTOK_Nepali_64K import PyHimalayanTOK_Nepali_64K

# ------------------------------------------------------------
# Configuration – adjust these paths as needed
VOCAB_TSV = "vocab_nepbpe/nepbpe_vocab_bilingual_v3.tsv"
FOLDING_RULES = [("सङ्ग", "संग"), ("सँग", "संग")]
SPACE_PIECE = "\u0120"   # U+0120 is the byte-alphabet surface for space (0x20)

# Sample sentences – you can add/remove any
SAMPLES = [
    "kumardahal536@gmail.com",
    "तिम्रो मुस्कानमा बिहानको उज्यालो भेटेँ",
    "तिम्रो आँखामा आफ्नै संसार देखेँ",
    "शब्दले भन्न नसक्ने भावना",
    "मुटुले चुपचाप तिमीलाई लेखेँ",
    "हावाले तिम्रो नाम बिस्तारै बोलाउँछ",
    "चन्द्रमाले तिम्रो यादमा रात सजाउँछ",
    "टाढा भए पनि मन नजिकै रहन्छ",
    "साँचो माया समयसँग कहिल्यै नहराउँछ",
    "तिमीसँग बितेको प्रत्येक पल",
    "जीवनको सबैभन्दा सुन्दर गीत बन्यो",
    "दुःखका बादल आए पनि",
    "तिम्रो साथले हरेक आँसु मुस्कान बन्यो",
    "माया भनेको केवल शब्द होइन",
    "एकअर्काको सपना बोक्ने यात्रा हो",
    "विश्वास, सम्मान र साथको डोरीले",
    "दुई आत्मालाई सधैं जोड्ने कथा हो",
    "यदि अर्को जन्मको कथा लेखियो भने",
    "फेरि पनि तिमी नै मेरो रोजाइ हुनेछौ",
    "यस जन्मझैं, त्यो जन्ममा पनि",
    "मेरो हरेक प्रार्थनाको उत्तर तिमी नै हुनेछौ",
    "नेपाल (आधिकारिक नाम: सङ्घीय लोकतान्त्रिक गणतन्त्र नेपाल)",
    "we are venome",
    "the study of mathematics in Nepal",
    "the quick brown fox jumps over the lazy dog",
    "a journey of a thousand miles begins with a single step",
    "to be or not to be that is the question",
    "Battalion commanders coordinated the offensive",
    "xylophone and psychology are fascinating subjects",
    "the chemical formula for water is H2O",
    "the year 2024 is almost over",
    "she scored 99.8% on the final exam",
    "नेपालको history धेरै ancient छ",
    "काठमाडौं is the capital city of Nepal",
    "UNIFIL completed its mission in Nepal",
    "the CEO of AI4Bharat spoke at GES 2025",
]
# ------------------------------------------------------------


def test_tokenizer(verbose=False, test_file=None):
    """
    Run a full test of the Himalayan tokenizer.

    Parameters
    ----------
    verbose : bool
        If True, print the token surfaces for each sample sentence.
    test_file : str or None
        If provided, path to a text file (one sentence per line) for fertility analysis.
    """
    # Load tokenizer
    tok = PyHimalayanTOK_Nepali_64K(folding_rules=FOLDING_RULES)
    n = tok.load_vocab_tsv(VOCAB_TSV)
    print(f"Loaded {n} tokens from {VOCAB_TSV}\n")

    # ------------------------------------------------------------------
    # 1) Sample sentences
    # ------------------------------------------------------------------
    sample_stats = []  # (tokens, spaces, content, ok)
    fails = 0

    print("=== Sentence‑level sample tokenization ===\n")
    for idx, sentence in enumerate(SAMPLES, 1):
        # Encode / decode
        ids = tok.encode(sentence)
        decoded = tok.decode(ids)
        norm = tok.normalize(sentence)
        ok = (decoded == norm)

        # Surfaces
        surfaces = [tok.get_token_surface(i) for i in ids]
        spaces = sum(1 for p in surfaces if p == SPACE_PIECE)
        content = len(ids) - spaces

        sample_stats.append((len(ids), spaces, content, ok))
        if not ok:
            fails += 1

        # Print details
        status = "OK" if ok else "FAIL"
        print(f"[{idx:3d}] Tokens: {len(ids):4d}  (content={content:3d}, spaces={spaces:2d})  {status}")
        # Show the sentence (truncated if long)
        disp = sentence if len(sentence) <= 70 else sentence[:67] + "..."
        print(f"      Sentence: {disp}")

        if verbose:
            # Show token surfaces (first 20, then ...)
            if len(surfaces) > 20:
                shown = " ".join(surfaces[:20]) + " ..."
            else:
                shown = " ".join(surfaces)
            print(f"      Tokens  : {shown}")

        # Always show decoded
        print(f"      Decoded : {decoded}")
        if not ok:
            print(f"      Expected: {norm}")
        print()  # blank line

    # Summary for samples
    if sample_stats:
        tokens_list = [s[0] for s in sample_stats]
        spaces_list = [s[1] for s in sample_stats]
        content_list = [s[2] for s in sample_stats]
        print("=== Sample summary ===")
        print(f"  Tokens/sentence  : mean={statistics.mean(tokens_list):.2f}  median={statistics.median(tokens_list):.2f}")
        print(f"  Content/sentence : mean={statistics.mean(content_list):.2f}  median={statistics.median(content_list):.2f}")
        print(f"  Spaces/sentence  : mean={statistics.mean(spaces_list):.2f}  median={statistics.median(spaces_list):.2f}")
        print(f"  Round‑trip       : {len(SAMPLES) - fails}/{len(SAMPLES)} OK\n")

    # ------------------------------------------------------------------
    # 2) Fertility over a large file (optional)
    # ------------------------------------------------------------------
    if test_file:
        print(f"=== Fertility analysis on '{test_file}' ===")
        space_id = tok.vocab_get_id(SPACE_PIECE)  # may be None
        total_tokens = 0
        total_spaces = 0
        total_sentences = 0
        t0 = time.perf_counter()

        try:
            with open(test_file, encoding="utf-8") as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue

                    if line_num % 1000 == 0:
                        elapsed = time.perf_counter() - t0
                        print(f"  ... line {line_num}: {total_tokens} tokens in {elapsed:.1f}s "
                              f"({total_tokens / max(1, elapsed):.0f} tok/s)", flush=True)

                    ids = tok.encode(line)
                    sp = ids.count(space_id) if space_id is not None else 0
                    total_tokens += len(ids)
                    total_spaces += sp
                    total_sentences += 1

                    if total_sentences >= 20000:
                        print(f"  Reached {total_sentences} lines limit", flush=True)
                        break

        except FileNotFoundError:
            print(f"  Error: File '{test_file}' not found. Skipping.")
            return
        except KeyboardInterrupt:
            print(f"\n  Interrupted after {total_sentences} lines", flush=True)
            return

        dt = time.perf_counter() - t0
        content_tokens = total_tokens - total_spaces
        print(f"  Sentences        : {total_sentences}")
        print(f"  Total tokens     : {total_tokens}")
        print(f"  Tokens/sentence  : {total_tokens / max(1, total_sentences):.3f}")
        print(f"  Content/sentence : {content_tokens / max(1, total_sentences):.3f} (excl. space tokens)")
        print(f"  Space fraction   : {total_spaces / max(1, total_tokens):.3f}")
        print(f"  Time             : {dt:.1f}s  ({total_tokens / max(1, dt):.0f} tok/s)")

In [20]:
test_tokenizer()

Loaded 64001 tokens from vocab_nepbpe/nepbpe_vocab_bilingual_v3.tsv

=== Sentence‑level sample tokenization ===

[  1] Tokens:    9  (content=  9, spaces= 0)  OK
      Sentence: kumardahal536@gmail.com
      Decoded : kumardahal536@gmail.com

[  2] Tokens:    7  (content=  7, spaces= 0)  OK
      Sentence: तिम्रो मुस्कानमा बिहानको उज्यालो भेटेँ
      Decoded : तिम्रो मुस्कानमा बिहानको उज्यालो भेटेँ

[  3] Tokens:    5  (content=  5, spaces= 0)  OK
      Sentence: तिम्रो आँखामा आफ्नै संसार देखेँ
      Decoded : तिम्रो आँखामा आफ्नै संसार देखेँ

[  4] Tokens:    4  (content=  4, spaces= 0)  OK
      Sentence: शब्दले भन्न नसक्ने भावना
      Decoded : शब्दले भन्न नसक्ने भावना

[  5] Tokens:    6  (content=  6, spaces= 0)  OK
      Sentence: मुटुले चुपचाप तिमीलाई लेखेँ
      Decoded : मुटुले चुपचाप तिमीलाई लेखेँ

[  6] Tokens:    6  (content=  6, spaces= 0)  OK
      Sentence: हावाले तिम्रो नाम बिस्तारै बोलाउँछ
      Decoded : हावाले तिम्रो नाम बिस्तारै बोलाउँछ

[  7] Tokens:    7  (content=

In [24]:
import os
from transformers import PreTrainedTokenizer
from HimalayanTOK_Nepali_64K import PyHimalayanTOK_Nepali_64K

class HimalayanTokenizer(PreTrainedTokenizer):
    """
    Hugging Face tokenizer wrapper for the Rust-based HimalayanTOK_Nepali_64K.
    """
    def __init__(
        self,
        vocab_file=None,
        unk_token="[UNK]",
        cls_token="[CLS]",
        sep_token="[SEP]",
        pad_token="[PAD]",
        mask_token="[MASK]",
        **kwargs
    ):
        # Create the Rust tokenizer FIRST (before any parent method that might use it)
        self.rust_tokenizer = PyHimalayanTOK_Nepali_64K()

        # Now call the parent initializer, passing special token arguments
        super().__init__(
            unk_token=unk_token,
            cls_token=cls_token,
            sep_token=sep_token,
            pad_token=pad_token,
            mask_token=mask_token,
            **kwargs
        )

        # Load the vocab file if provided
        if vocab_file is not None:
            self.rust_tokenizer.load_vocab_tsv(vocab_file)

    def _tokenize(self, text):
        return self.rust_tokenizer.tokenize_to_strings(text)

    def _convert_token_to_id(self, token):
        return self.rust_tokenizer.vocab_get_id(token)

    def _convert_id_to_token(self, index):
        return self.rust_tokenizer.get_token_surface(index)

    def get_vocab(self):
        return self.rust_tokenizer.get_vocab_dict()

    def save_vocabulary(self, save_directory, filename_prefix=None):
        if not os.path.exists(save_directory):
            os.makedirs(save_directory)
        prefix = filename_prefix or ""
        vocab_file = os.path.join(save_directory, f"{prefix}vocab.tsv")
        self.rust_tokenizer.save_vocab_tsv(vocab_file)
        return (vocab_file,)

    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        cls = [self.cls_token_id]
        sep = [self.sep_token_id]
        if token_ids_1 is None:
            return cls + token_ids_0 + sep
        else:
            return cls + token_ids_0 + sep + token_ids_1 + sep

    def get_special_tokens_mask(
        self, token_ids_0, token_ids_1=None, already_has_special_tokens=False
    ):
        if already_has_special_tokens:
            return super().get_special_tokens_mask(
                token_ids_0, token_ids_1, already_has_special_tokens
            )
        if token_ids_1 is None:
            return [1] + [0] * len(token_ids_0) + [1]
        else:
            return [1] + [0] * len(token_ids_0) + [1] + [0] * len(token_ids_1) + [1]

    def create_token_type_ids_from_sequences(self, token_ids_0, token_ids_1=None):
        sep = [self.sep_token_id]
        cls = [self.cls_token_id]
        if token_ids_1 is None:
            return len(cls + token_ids_0 + sep) * [0]
        else:
            return len(cls + token_ids_0 + sep) * [0] + len(token_ids_1 + sep) * [1]

In [30]:
# Make sure you have the corrected wrapper class defined (as in previous message)
tokenizer = HimalayanTokenizer(vocab_file="vocab_nepbpe/nepbpe_vocab_bilingual_v3.tsv")

def show_tokenization(text):
    ids = tokenizer.encode(text, add_special_tokens=True)
    tokens = tokenizer.convert_ids_to_tokens(ids)
    joined_tokens = " ".join(tokens)

    decoded = tokenizer.rust_tokenizer.decode(ids)
    normalized = tokenizer.rust_tokenizer.normalize(text)
    ok = (decoded == normalized)

    print(f"text:    {text}")
    print(f"ids:     {ids}")
    print(f"tokens:  {tokens}")
    print(f"decoded: {joined_tokens}")
    print(f"roundtrip: {'✅ OK' if ok else '❌ FAIL'}")
    if not ok:
        print(f"  decoded (true): {decoded!r}")
        print(f"  normalized:     {normalized!r}")
    print()

# Test
show_tokenization("नेपालको संविधान २०७२ मा जारी भएको थियो")

text:    नेपालको संविधान २०७२ मा जारी भएको थियो
ids:     [3, 1622, 1607, 4664, 810, 1725, 857, 938, 1]
tokens:  ['[CLS]', '▁नेपालको', '▁संविधान', '▁२०७२', '▁मा', '▁जारी', '▁भएको', '▁थियो', '[SEP]']
decoded: [CLS] ▁नेपालको ▁संविधान ▁२०७२ ▁मा ▁जारी ▁भएको ▁थियो [SEP]
roundtrip: ❌ FAIL
  decoded (true): 'ः नेपालको संविधान २०७२ मा जारी भएको थियोँ'
  normalized:     'नेपालको संविधान २०७२ मा जारी भएको थियो'

